# Лабораторная работа 6.2  
## Kubernetes и canary deployment

Тема: развёртывание ML inference-сервиса в Kubernetes: HPA, canary- и A/B-деплой, rollback.

Этот notebook собирает весь программный код лабораторной работы в один `.ipynb`-файл.

После выполнения ячеек будет создан проект со следующими компонентами:

- FastAPI inference-сервис;
- Dockerfile;
- Kubernetes-манифесты;
- Deployment v1 и v2;
- Service для общего доступа и A/B-тестирования;
- ConfigMap;
- Secret;
- readinessProbe и livenessProbe;
- resource requests и limits;
- Horizontal Pod Autoscaler;
- скрипты сборки, деплоя и rollback;
- скрипты нагрузочного и A/B-тестирования;
- README и шаблон REPORT.

In [ ]:
# Создаём структуру проекта лабораторной работы.

from pathlib import Path

directories = [
    "app",
    "k8s",
    "scripts",
    "load_tests",
    "results",
]

for directory in directories:
    Path(directory).mkdir(parents=True, exist_ok=True)

Path("app/__init__.py").write_text("", encoding="utf-8")

print("Структура проекта создана.")

## 1. Зависимости проекта

Сервис реализован на FastAPI.

В данной лабораторной работе используется демонстрационный inference-сервис без внешнего ML-фреймворка.  
Это упрощает развёртывание в Kubernetes и позволяет сконцентрироваться на HPA, canary, A/B и rollback.

Предсказание имитирует кредитный скоринг на основе нескольких признаков.

In [ ]:
# Файл зависимостей для inference-сервиса.

Path("requirements.txt").write_text(r'''
fastapi==0.115.6
uvicorn[standard]==0.34.0
pydantic==2.10.4
'''.strip() + "\n", encoding="utf-8")

print("requirements.txt создан.")

## 2. FastAPI inference-сервис

Сервис поддерживает endpoints:

- `GET /health`
- `GET /metadata`
- `POST /predict`

Версии v1 и v2 различаются через переменные окружения:

- `SERVICE_VERSION`
- `MODEL_VERSION`
- `PREDICTION_THRESHOLD`
- `ARTIFICIAL_DELAY_MS`
- `BROKEN_MODE`

`BROKEN_MODE=true` можно использовать для имитации ошибки и проверки rollback.

In [ ]:
# Основной код FastAPI-приложения.

Path("app/main.py").write_text(r'''
import os
import time
from typing import Literal

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, ConfigDict, Field


# -----------------------------------------------------------------------------
# Конфигурация сервиса.
# Значения задаются через переменные окружения в Kubernetes Deployment.
# -----------------------------------------------------------------------------

SERVICE_NAME = os.getenv("SERVICE_NAME", "ml-service")
SERVICE_VERSION = os.getenv("SERVICE_VERSION", "v1")
MODEL_NAME = os.getenv("MODEL_NAME", "credit_scoring_model")
MODEL_VERSION = os.getenv("MODEL_VERSION", "1.0.0")

# Threshold отличается между версиями модели и влияет на предсказание.
PREDICTION_THRESHOLD = float(os.getenv("PREDICTION_THRESHOLD", "0.50"))

# Искусственная задержка помогает демонстрировать разницу latency между v1 и v2.
ARTIFICIAL_DELAY_MS = int(os.getenv("ARTIFICIAL_DELAY_MS", "0"))

# Если BROKEN_MODE=true, endpoint /predict будет возвращать ошибку.
# Это удобно для демонстрации rollback.
BROKEN_MODE = os.getenv("BROKEN_MODE", "false").lower() == "true"

# Небольшая CPU-нагрузка для экспериментов с HPA.
# Значение можно увеличить в ConfigMap или Deployment.
CPU_BURN_ITERATIONS = int(os.getenv("CPU_BURN_ITERATIONS", "50000"))


app = FastAPI(
    title="ML Inference Service",
    description="Demo ML inference-service for Kubernetes, HPA, canary and A/B deployment.",
    version=SERVICE_VERSION,
)


class PredictionRequest(BaseModel):
    # extra='forbid' запрещает лишние поля в JSON.
    model_config = ConfigDict(extra="forbid")

    age: int = Field(..., ge=18, le=100)
    income: float = Field(..., gt=0)
    loan_amount: float = Field(..., gt=0)
    employment_years: float = Field(..., ge=0, le=60)


class PredictionResponse(BaseModel):
    prediction: Literal[0, 1]
    probability: float
    model_name: str
    model_version: str
    service_version: str
    threshold: float
    latency_ms: float


def burn_cpu(iterations: int) -> None:
    # Имитируем CPU-bound часть inference.
    # Это помогает HPA увидеть рост CPU utilization при нагрузке.
    value = 0
    for i in range(iterations):
        value += (i * i) % 97
    return None


def calculate_probability(payload: PredictionRequest) -> float:
    # Упрощённая демонстрационная формула кредитного риска.
    # Чем выше отношение кредита к доходу и меньше стаж, тем выше риск.
    debt_to_income = payload.loan_amount / max(payload.income, 1)
    age_factor = max(0, 45 - payload.age) / 100
    employment_factor = max(0, 10 - payload.employment_years) / 20

    raw_score = (
        0.15
        + 0.12 * debt_to_income
        + 0.35 * age_factor
        + 0.25 * employment_factor
    )

    # Версия v2 немного отличается от v1.
    # Это позволяет увидеть разные ответы при canary и A/B.
    if SERVICE_VERSION == "v2":
        raw_score += 0.05

    probability = max(0.0, min(1.0, raw_score))
    return probability


@app.get("/health")
def health():
    # readinessProbe и livenessProbe будут обращаться к этому endpoint.
    if BROKEN_MODE:
        raise HTTPException(
            status_code=503,
            detail="Service is in BROKEN_MODE",
        )

    return {
        "status": "ok",
        "service": SERVICE_NAME,
        "service_version": SERVICE_VERSION,
        "model_loaded": True,
    }


@app.get("/metadata")
def metadata():
    # Endpoint возвращает данные о модели и версии сервиса.
    return {
        "model_name": MODEL_NAME,
        "model_version": MODEL_VERSION,
        "service_version": SERVICE_VERSION,
        "framework": "demo-python",
        "task_type": "binary_classification",
        "features": [
            "age",
            "income",
            "loan_amount",
            "employment_years",
        ],
        "threshold": PREDICTION_THRESHOLD,
        "broken_mode": BROKEN_MODE,
        "artificial_delay_ms": ARTIFICIAL_DELAY_MS,
        "cpu_burn_iterations": CPU_BURN_ITERATIONS,
    }


@app.post("/predict", response_model=PredictionResponse)
def predict(payload: PredictionRequest):
    start_time = time.perf_counter()

    if BROKEN_MODE:
        raise HTTPException(
            status_code=500,
            detail="Artificial inference error for rollback demonstration",
        )

    # Искусственная задержка для демонстрации деградации latency.
    if ARTIFICIAL_DELAY_MS > 0:
        time.sleep(ARTIFICIAL_DELAY_MS / 1000)

    burn_cpu(CPU_BURN_ITERATIONS)

    probability = calculate_probability(payload)
    prediction = 1 if probability >= PREDICTION_THRESHOLD else 0

    latency_ms = round((time.perf_counter() - start_time) * 1000, 3)

    return {
        "prediction": prediction,
        "probability": round(probability, 6),
        "model_name": MODEL_NAME,
        "model_version": MODEL_VERSION,
        "service_version": SERVICE_VERSION,
        "threshold": PREDICTION_THRESHOLD,
        "latency_ms": latency_ms,
    }
'''.strip() + "\n", encoding="utf-8")

print("app/main.py создан.")

## 3. Dockerfile

Один Dockerfile используется для обеих версий сервиса.

Версии `v1` и `v2` задаются через переменные окружения в Kubernetes Deployment.  
При необходимости можно собрать два тега одного и того же образа:

- `ml-service:v1`
- `ml-service:v2`

In [ ]:
# Dockerfile для контейнеризации inference-сервиса.

Path("Dockerfile").write_text(r'''
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir --upgrade pip     && pip install --no-cache-dir -r requirements.txt

COPY app ./app

EXPOSE 8000

# В Kubernetes контейнер должен слушать 0.0.0.0, а не 127.0.0.1.
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
'''.strip() + "\n", encoding="utf-8")

Path(".dockerignore").write_text(r'''
__pycache__/
*.pyc
.ipynb_checkpoints/
.git/
.venv/
venv/
results/
'''.strip() + "\n", encoding="utf-8")

print("Dockerfile и .dockerignore созданы.")

## 4. Kubernetes-манифесты

Будут созданы следующие YAML-файлы:

- `namespace.yaml`
- `configmap.yaml`
- `secret.yaml`
- `deployment-v1.yaml`
- `deployment-v2.yaml`
- `service.yaml`
- `service-v1.yaml`
- `service-v2.yaml`
- `hpa.yaml`

Общий `service.yaml` выбирает pods по label `app: ml-service`, поэтому может направлять трафик и на v1, и на v2.  
Это используется для canary deployment.

Отдельные `service-v1.yaml` и `service-v2.yaml` используются для A/B deployment.

In [ ]:
# Namespace для изоляции ресурсов лабораторной работы.

Path("k8s/namespace.yaml").write_text(r'''
apiVersion: v1
kind: Namespace
metadata:
  name: ml-lab
'''.strip() + "\n", encoding="utf-8")

print("k8s/namespace.yaml создан.")

In [ ]:
# ConfigMap хранит несекретные параметры приложения.

Path("k8s/configmap.yaml").write_text(r'''
apiVersion: v1
kind: ConfigMap
metadata:
  name: ml-service-config
  namespace: ml-lab
data:
  SERVICE_NAME: "ml-service"
  MODEL_NAME: "credit_scoring_model"
  CPU_BURN_ITERATIONS: "70000"
'''.strip() + "\n", encoding="utf-8")

print("k8s/configmap.yaml создан.")

In [ ]:
# Secret имитирует секретные параметры.
# Значения приведены только для учебных целей.
# В production нельзя хранить реальные секреты в открытом виде.

Path("k8s/secret.yaml").write_text(r'''
apiVersion: v1
kind: Secret
metadata:
  name: ml-service-secret
  namespace: ml-lab
type: Opaque
stringData:
  API_TOKEN: "demo-token-do-not-use-in-production"
'''.strip() + "\n", encoding="utf-8")

print("k8s/secret.yaml создан.")

In [ ]:
# Deployment стабильной версии v1.
# HPA будет масштабировать именно этот Deployment.

Path("k8s/deployment-v1.yaml").write_text(r'''
apiVersion: apps/v1
kind: Deployment
metadata:
  name: ml-service-v1
  namespace: ml-lab
  labels:
    app: ml-service
    version: v1
spec:
  replicas: 2
  selector:
    matchLabels:
      app: ml-service
      version: v1
  template:
    metadata:
      labels:
        app: ml-service
        version: v1
    spec:
      containers:
        - name: ml-service
          image: ml-service:v1
          imagePullPolicy: IfNotPresent
          ports:
            - containerPort: 8000
          env:
            - name: SERVICE_NAME
              valueFrom:
                configMapKeyRef:
                  name: ml-service-config
                  key: SERVICE_NAME
            - name: MODEL_NAME
              valueFrom:
                configMapKeyRef:
                  name: ml-service-config
                  key: MODEL_NAME
            - name: CPU_BURN_ITERATIONS
              valueFrom:
                configMapKeyRef:
                  name: ml-service-config
                  key: CPU_BURN_ITERATIONS
            - name: SERVICE_VERSION
              value: "v1"
            - name: MODEL_VERSION
              value: "1.0.0"
            - name: PREDICTION_THRESHOLD
              value: "0.50"
            - name: ARTIFICIAL_DELAY_MS
              value: "0"
            - name: BROKEN_MODE
              value: "false"
            - name: API_TOKEN
              valueFrom:
                secretKeyRef:
                  name: ml-service-secret
                  key: API_TOKEN
          resources:
            requests:
              cpu: "100m"
              memory: "128Mi"
            limits:
              cpu: "500m"
              memory: "512Mi"
          readinessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 5
            periodSeconds: 5
            timeoutSeconds: 2
            failureThreshold: 3
          livenessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 15
            periodSeconds: 10
            timeoutSeconds: 2
            failureThreshold: 3
'''.strip() + "\n", encoding="utf-8")

print("k8s/deployment-v1.yaml создан.")

In [ ]:
# Deployment новой версии v2.
# По умолчанию реплик 1, чтобы использовать v2 как canary.

Path("k8s/deployment-v2.yaml").write_text(r'''
apiVersion: apps/v1
kind: Deployment
metadata:
  name: ml-service-v2
  namespace: ml-lab
  labels:
    app: ml-service
    version: v2
spec:
  replicas: 1
  selector:
    matchLabels:
      app: ml-service
      version: v2
  template:
    metadata:
      labels:
        app: ml-service
        version: v2
    spec:
      containers:
        - name: ml-service
          image: ml-service:v2
          imagePullPolicy: IfNotPresent
          ports:
            - containerPort: 8000
          env:
            - name: SERVICE_NAME
              valueFrom:
                configMapKeyRef:
                  name: ml-service-config
                  key: SERVICE_NAME
            - name: MODEL_NAME
              valueFrom:
                configMapKeyRef:
                  name: ml-service-config
                  key: MODEL_NAME
            - name: CPU_BURN_ITERATIONS
              valueFrom:
                configMapKeyRef:
                  name: ml-service-config
                  key: CPU_BURN_ITERATIONS
            - name: SERVICE_VERSION
              value: "v2"
            - name: MODEL_VERSION
              value: "2.0.0"
            - name: PREDICTION_THRESHOLD
              value: "0.45"
            - name: ARTIFICIAL_DELAY_MS
              value: "20"
            - name: BROKEN_MODE
              value: "false"
            - name: API_TOKEN
              valueFrom:
                secretKeyRef:
                  name: ml-service-secret
                  key: API_TOKEN
          resources:
            requests:
              cpu: "100m"
              memory: "128Mi"
            limits:
              cpu: "500m"
              memory: "512Mi"
          readinessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 5
            periodSeconds: 5
            timeoutSeconds: 2
            failureThreshold: 3
          livenessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 15
            periodSeconds: 10
            timeoutSeconds: 2
            failureThreshold: 3
'''.strip() + "\n", encoding="utf-8")

print("k8s/deployment-v2.yaml создан.")

In [ ]:
# Общий Service для canary deployment.
# Он выбирает все pods с label app=ml-service, независимо от version.

Path("k8s/service.yaml").write_text(r'''
apiVersion: v1
kind: Service
metadata:
  name: ml-service
  namespace: ml-lab
spec:
  type: NodePort
  selector:
    app: ml-service
  ports:
    - name: http
      port: 80
      targetPort: 8000
      nodePort: 30080
'''.strip() + "\n", encoding="utf-8")

print("k8s/service.yaml создан.")

In [ ]:
# Отдельный Service для версии v1.
# Используется для A/B deployment.

Path("k8s/service-v1.yaml").write_text(r'''
apiVersion: v1
kind: Service
metadata:
  name: ml-service-v1
  namespace: ml-lab
spec:
  type: NodePort
  selector:
    app: ml-service
    version: v1
  ports:
    - name: http
      port: 80
      targetPort: 8000
      nodePort: 30081
'''.strip() + "\n", encoding="utf-8")

# Отдельный Service для версии v2.
Path("k8s/service-v2.yaml").write_text(r'''
apiVersion: v1
kind: Service
metadata:
  name: ml-service-v2
  namespace: ml-lab
spec:
  type: NodePort
  selector:
    app: ml-service
    version: v2
  ports:
    - name: http
      port: 80
      targetPort: 8000
      nodePort: 30082
'''.strip() + "\n", encoding="utf-8")

print("k8s/service-v1.yaml и k8s/service-v2.yaml созданы.")

In [ ]:
# Horizontal Pod Autoscaler.
# Масштабирует Deployment v1 от 2 до 5 реплик по CPU utilization.

Path("k8s/hpa.yaml").write_text(r'''
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: ml-service-hpa
  namespace: ml-lab
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: ml-service-v1
  minReplicas: 2
  maxReplicas: 5
  metrics:
    - type: Resource
      resource:
        name: cpu
        target:
          type: Utilization
          averageUtilization: 50
'''.strip() + "\n", encoding="utf-8")

print("k8s/hpa.yaml создан.")

## 5. Скрипты сборки, деплоя и rollback

Скрипты находятся в каталоге `scripts/`.

Перед запуском сделайте их исполняемыми:

    chmod +x scripts/*.sh

In [ ]:
# Скрипт сборки Docker-образов v1 и v2.

Path("scripts/build_images.sh").write_text(r'''
#!/usr/bin/env bash
set -euo pipefail

echo "Building Docker images..."

docker build -t ml-service:v1 .
docker build -t ml-service:v2 .

echo "Images built:"
docker images | grep ml-service || true

echo ""
echo "If you use minikube, run:"
echo "  minikube image load ml-service:v1"
echo "  minikube image load ml-service:v2"
echo ""
echo "If you use kind, run:"
echo "  kind load docker-image ml-service:v1"
echo "  kind load docker-image ml-service:v2"
'''.strip() + "\n", encoding="utf-8")

print("scripts/build_images.sh создан.")

In [ ]:
# Скрипт базового деплоя в Kubernetes.

Path("scripts/deploy.sh").write_text(r'''
#!/usr/bin/env bash
set -euo pipefail

echo "Applying Kubernetes manifests..."

kubectl apply -f k8s/namespace.yaml
kubectl apply -f k8s/configmap.yaml
kubectl apply -f k8s/secret.yaml
kubectl apply -f k8s/deployment-v1.yaml
kubectl apply -f k8s/service.yaml
kubectl apply -f k8s/service-v1.yaml
kubectl apply -f k8s/hpa.yaml

echo "Waiting for v1 rollout..."
kubectl rollout status deployment/ml-service-v1 -n ml-lab

echo "Current resources:"
kubectl get all -n ml-lab
'''.strip() + "\n", encoding="utf-8")

print("scripts/deploy.sh создан.")

In [ ]:
# Скрипт деплоя canary v2.

Path("scripts/deploy_canary.sh").write_text(r'''
#!/usr/bin/env bash
set -euo pipefail

echo "Deploying canary version v2..."

kubectl apply -f k8s/deployment-v2.yaml
kubectl apply -f k8s/service-v2.yaml

kubectl rollout status deployment/ml-service-v2 -n ml-lab

echo "Canary deployed."
echo "Current pods:"
kubectl get pods -n ml-lab -l app=ml-service -o wide
'''.strip() + "\n", encoding="utf-8")

print("scripts/deploy_canary.sh создан.")

In [ ]:
# Скрипт изменения доли canary-трафика через количество реплик.
# Пример:
#   scripts/set_canary_ratio.sh 7 3

Path("scripts/set_canary_ratio.sh").write_text(r'''
#!/usr/bin/env bash
set -euo pipefail

V1_REPLICAS="${1:-9}"
V2_REPLICAS="${2:-1}"

echo "Setting canary ratio:"
echo "  v1 replicas: ${V1_REPLICAS}"
echo "  v2 replicas: ${V2_REPLICAS}"

kubectl scale deployment ml-service-v1 -n ml-lab --replicas="${V1_REPLICAS}"
kubectl scale deployment ml-service-v2 -n ml-lab --replicas="${V2_REPLICAS}"

kubectl rollout status deployment/ml-service-v1 -n ml-lab
kubectl rollout status deployment/ml-service-v2 -n ml-lab

kubectl get pods -n ml-lab -l app=ml-service
'''.strip() + "\n", encoding="utf-8")

print("scripts/set_canary_ratio.sh создан.")

In [ ]:
# Скрипт rollback.
# При схеме с двумя Deployment rollback выполняется масштабированием v2 до 0.

Path("scripts/rollback.sh").write_text(r'''
#!/usr/bin/env bash
set -euo pipefail

echo "Rolling back to stable v1..."

kubectl scale deployment ml-service-v2 -n ml-lab --replicas=0 || true
kubectl scale deployment ml-service-v1 -n ml-lab --replicas=3

kubectl rollout status deployment/ml-service-v1 -n ml-lab

echo "Rollback completed."
kubectl get pods -n ml-lab -l app=ml-service
'''.strip() + "\n", encoding="utf-8")

print("scripts/rollback.sh создан.")

In [ ]:
# Скрипт для имитации сломанной версии v2.
# Он устанавливает BROKEN_MODE=true в Deployment v2.
# После этого /health и /predict начнут возвращать ошибки.

Path("scripts/break_v2.sh").write_text(r'''
#!/usr/bin/env bash
set -euo pipefail

echo "Enabling BROKEN_MODE for v2..."

kubectl set env deployment/ml-service-v2 -n ml-lab BROKEN_MODE=true
kubectl rollout status deployment/ml-service-v2 -n ml-lab

echo "v2 is now broken. Use scripts/rollback.sh to rollback."
'''.strip() + "\n", encoding="utf-8")

print("scripts/break_v2.sh создан.")

## 6. Нагрузочное тестирование

В лабораторной работе можно использовать `hey`, `wrk`, `k6` или собственный Python-скрипт.

Ниже создаются:

- `load_tests/hey-commands.md`
- `load_tests/k6-test.js`
- `scripts/canary_check.py`
- `scripts/ab_test_client.py`

In [ ]:
# Команды hey для нагрузочного тестирования.

Path("load_tests/hey-commands.md").write_text(r'''
# Нагрузочное тестирование через hey

## Проверка общего Service

Если используется NodePort:

    hey -z 3m -c 50 -m POST       -H "Content-Type: application/json"       -d '{"age":35,"income":85000,"loan_amount":300000,"employment_years":7}'       http://localhost:30080/predict

Для minikube можно получить URL:

    minikube service ml-service -n ml-lab --url

Затем использовать полученный URL:

    hey -z 3m -c 50 -m POST       -H "Content-Type: application/json"       -d '{"age":35,"income":85000,"loan_amount":300000,"employment_years":7}'       http://<service-url>/predict

## Наблюдение за HPA

В отдельных терминалах:

    kubectl get hpa -n ml-lab -w
    kubectl get pods -n ml-lab -w
    kubectl top pods -n ml-lab
    kubectl describe hpa ml-service-hpa -n ml-lab
'''.strip() + "\n", encoding="utf-8")

print("load_tests/hey-commands.md создан.")

In [ ]:
# k6-тест для нагрузки на endpoint /predict.

Path("load_tests/k6-test.js").write_text(r'''
import http from "k6/http";
import { check, sleep } from "k6";

export const options = {
  stages: [
    { duration: "30s", target: 10 },
    { duration: "1m", target: 50 },
    { duration: "30s", target: 0 },
  ],
};

const BASE_URL = __ENV.BASE_URL || "http://localhost:30080";

export default function () {
  const payload = JSON.stringify({
    age: 35,
    income: 85000,
    loan_amount: 300000,
    employment_years: 7,
  });

  const params = {
    headers: {
      "Content-Type": "application/json",
    },
  };

  const response = http.post(`${BASE_URL}/predict`, payload, params);

  check(response, {
    "status is 200": (r) => r.status === 200,
    "has service_version": (r) => r.body.includes("service_version"),
  });

  sleep(0.1);
}
'''.strip() + "\n", encoding="utf-8")

print("load_tests/k6-test.js создан.")

In [ ]:
# Скрипт проверки canary-распределения.
# Он отправляет запросы в общий Service и считает, сколько ответов пришло от v1 и v2.

Path("scripts/canary_check.py").write_text(r'''
import csv
import json
import os
import time
import urllib.request
from pathlib import Path


URL = os.getenv("SERVICE_URL", "http://localhost:30080")
REQUESTS = int(os.getenv("REQUESTS", "200"))

PAYLOAD = {
    "age": 35,
    "income": 85000,
    "loan_amount": 300000,
    "employment_years": 7,
}

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
RESULTS_PATH = RESULTS_DIR / "canary-results.csv"


def post_json(url: str, payload: dict) -> tuple[int, dict, float]:
    data = json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(
        url=url,
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )

    started_at = time.perf_counter()

    try:
        with urllib.request.urlopen(request, timeout=10) as response:
            body = response.read().decode("utf-8")
            latency_ms = (time.perf_counter() - started_at) * 1000
            return response.status, json.loads(body), latency_ms
    except Exception as exc:
        latency_ms = (time.perf_counter() - started_at) * 1000
        return 0, {"error": str(exc)}, latency_ms


def main() -> None:
    rows = []

    counts = {
        "v1": 0,
        "v2": 0,
        "errors": 0,
    }

    for i in range(REQUESTS):
        status, body, latency_ms = post_json(f"{URL}/predict", PAYLOAD)
        service_version = body.get("service_version", "error")

        if status == 200 and service_version in counts:
            counts[service_version] += 1
        else:
            counts["errors"] += 1

        rows.append(
            {
                "request_id": i + 1,
                "status": status,
                "service_version": service_version,
                "latency_ms": round(latency_ms, 3),
                "body": json.dumps(body, ensure_ascii=False),
            }
        )

    with RESULTS_PATH.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=["request_id", "status", "service_version", "latency_ms", "body"],
        )
        writer.writeheader()
        writer.writerows(rows)

    print("Canary check completed.")
    print(f"Results saved to: {RESULTS_PATH}")
    print(counts)

    total_ok = counts["v1"] + counts["v2"]
    if total_ok > 0:
        print(f"v1 share: {counts['v1'] / total_ok:.2%}")
        print(f"v2 share: {counts['v2'] / total_ok:.2%}")


if __name__ == "__main__":
    main()
'''.strip() + "\n", encoding="utf-8")

print("scripts/canary_check.py создан.")

In [ ]:
# Скрипт A/B-тестирования.
# Он отправляет часть запросов в Service v1, часть — в Service v2.

Path("scripts/ab_test_client.py").write_text(r'''
import csv
import json
import os
import statistics
import time
import urllib.request
from pathlib import Path


URL_A = os.getenv("SERVICE_URL_A", "http://localhost:30081")
URL_B = os.getenv("SERVICE_URL_B", "http://localhost:30082")
REQUESTS_PER_GROUP = int(os.getenv("REQUESTS_PER_GROUP", "100"))

PAYLOAD = {
    "age": 35,
    "income": 85000,
    "loan_amount": 300000,
    "employment_years": 7,
}

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
RESULTS_PATH = RESULTS_DIR / "ab-results.csv"


def post_json(url: str, payload: dict) -> tuple[int, dict, float]:
    data = json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(
        url=f"{url}/predict",
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )

    started_at = time.perf_counter()

    try:
        with urllib.request.urlopen(request, timeout=10) as response:
            body = response.read().decode("utf-8")
            latency_ms = (time.perf_counter() - started_at) * 1000
            return response.status, json.loads(body), latency_ms
    except Exception as exc:
        latency_ms = (time.perf_counter() - started_at) * 1000
        return 0, {"error": str(exc)}, latency_ms


def run_group(group: str, url: str) -> list[dict]:
    rows = []

    for i in range(REQUESTS_PER_GROUP):
        status, body, latency_ms = post_json(url, PAYLOAD)

        rows.append(
            {
                "group": group,
                "request_id": i + 1,
                "status": status,
                "service_version": body.get("service_version", "error"),
                "model_version": body.get("model_version", "error"),
                "prediction": body.get("prediction", ""),
                "probability": body.get("probability", ""),
                "latency_ms": round(latency_ms, 3),
            }
        )

    return rows


def summarize(rows: list[dict]) -> None:
    for group in ["A", "B"]:
        group_rows = [row for row in rows if row["group"] == group]
        latencies = [float(row["latency_ms"]) for row in group_rows]
        errors = [row for row in group_rows if int(row["status"]) != 200]
        predictions = [
            int(row["prediction"])
            for row in group_rows
            if str(row["prediction"]).isdigit()
        ]

        print(f"Group {group}")
        print(f"  requests: {len(group_rows)}")
        print(f"  errors: {len(errors)}")
        print(f"  error_rate: {len(errors) / len(group_rows):.2%}")
        print(f"  avg_latency_ms: {statistics.mean(latencies):.3f}")
        print(f"  max_latency_ms: {max(latencies):.3f}")

        if predictions:
            print(f"  avg_prediction: {statistics.mean(predictions):.3f}")

        versions = {}
        for row in group_rows:
            versions[row["service_version"]] = versions.get(row["service_version"], 0) + 1

        print(f"  versions: {versions}")


def main() -> None:
    rows = []
    rows.extend(run_group("A", URL_A))
    rows.extend(run_group("B", URL_B))

    with RESULTS_PATH.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=[
                "group",
                "request_id",
                "status",
                "service_version",
                "model_version",
                "prediction",
                "probability",
                "latency_ms",
            ],
        )
        writer.writeheader()
        writer.writerows(rows)

    print(f"A/B results saved to: {RESULTS_PATH}")
    summarize(rows)


if __name__ == "__main__":
    main()
'''.strip() + "\n", encoding="utf-8")

print("scripts/ab_test_client.py создан.")

## 7. README.md

README содержит инструкции для воспроизведения лабораторной работы.

In [ ]:
Path("README.md").write_text(r'''
# Лабораторная работа 6.2
## Kubernetes и canary deployment

Проект демонстрирует развёртывание ML inference-сервиса в Kubernetes.

Реализовано:

- FastAPI inference-сервис;
- Dockerfile;
- Kubernetes Deployment v1 и v2;
- Service;
- ConfigMap;
- Secret;
- readinessProbe;
- livenessProbe;
- resource requests и limits;
- HPA;
- canary deployment;
- A/B deployment;
- rollback;
- нагрузочное тестирование.

## 1. Локальный запуск

Установить зависимости:

    pip install -r requirements.txt

Запустить сервис:

    uvicorn app.main:app --host 0.0.0.0 --port 8000

Проверить:

    curl http://localhost:8000/health
    curl http://localhost:8000/metadata

Запрос predict:

    curl -X POST http://localhost:8000/predict       -H "Content-Type: application/json"       -d '{"age":35,"income":85000,"loan_amount":300000,"employment_years":7}'

## 2. Сборка Docker-образов

    chmod +x scripts/*.sh
    scripts/build_images.sh

Для minikube:

    minikube image load ml-service:v1
    minikube image load ml-service:v2

Для kind:

    kind load docker-image ml-service:v1
    kind load docker-image ml-service:v2

## 3. Деплой v1 в Kubernetes

    scripts/deploy.sh

Проверка:

    kubectl get all -n ml-lab
    kubectl get hpa -n ml-lab

Если используется minikube:

    minikube service ml-service -n ml-lab --url

## 4. Проверка endpoints

Через NodePort:

    curl http://localhost:30080/health
    curl http://localhost:30080/metadata

    curl -X POST http://localhost:30080/predict       -H "Content-Type: application/json"       -d '{"age":35,"income":85000,"loan_amount":300000,"employment_years":7}'

## 5. HPA

Убедиться, что установлен metrics-server.

Для minikube:

    minikube addons enable metrics-server

Наблюдение:

    kubectl get hpa -n ml-lab -w
    kubectl get pods -n ml-lab -w
    kubectl top pods -n ml-lab

Нагрузка через hey:

    hey -z 3m -c 50 -m POST       -H "Content-Type: application/json"       -d '{"age":35,"income":85000,"loan_amount":300000,"employment_years":7}'       http://localhost:30080/predict

Или через k6:

    BASE_URL=http://localhost:30080 k6 run load_tests/k6-test.js

## 6. Canary deployment

Развернуть v2:

    scripts/deploy_canary.sh

Установить долю примерно 10%:

    scripts/set_canary_ratio.sh 9 1

Проверить фактическую долю:

    SERVICE_URL=http://localhost:30080 REQUESTS=200 python scripts/canary_check.py

Установить долю примерно 30%:

    scripts/set_canary_ratio.sh 7 3

Повторить проверку:

    SERVICE_URL=http://localhost:30080 REQUESTS=200 python scripts/canary_check.py

## 7. A/B deployment

Для A/B используются отдельные Services:

- ml-service-v1 на NodePort 30081;
- ml-service-v2 на NodePort 30082.

Запуск клиента:

    SERVICE_URL_A=http://localhost:30081     SERVICE_URL_B=http://localhost:30082     REQUESTS_PER_GROUP=100     python scripts/ab_test_client.py

Результаты сохраняются в:

    results/ab-results.csv

## 8. Rollback

Имитация поломки v2:

    scripts/break_v2.sh

Rollback к v1:

    scripts/rollback.sh

Проверить:

    kubectl get pods -n ml-lab
    curl http://localhost:30080/metadata

## 9. Диагностика

Полезные команды:

    kubectl get pods -n ml-lab -o wide
    kubectl describe pod <pod-name> -n ml-lab
    kubectl logs <pod-name> -n ml-lab
    kubectl describe hpa ml-service-hpa -n ml-lab
    kubectl get events -n ml-lab --sort-by=.metadata.creationTimestamp

## 10. Очистка

    kubectl delete namespace ml-lab
'''.strip() + "\n", encoding="utf-8")

print("README.md создан.")

## 8. Шаблон отчёта REPORT.md

Фактические значения необходимо заполнить после выполнения экспериментов.

In [ ]:
Path("REPORT.md").write_text(r'''
# Отчёт по лабораторной работе 6.2
## Kubernetes и canary deployment

## 1. Титульная часть

- Название лабораторной работы: Kubernetes и canary deployment
- ФИО студента:
- Группа:
- Дата выполнения:
- Kubernetes-окружение:
- Используемый ML-сервис:
- Версия v1:
- Версия v2:

## 2. Описание inference-сервиса

Сервис реализован на FastAPI.

Endpoints:

- GET /health
- GET /metadata
- POST /predict

Версия v1:

- service_version: v1
- model_version: 1.0.0
- threshold: 0.50

Версия v2:

- service_version: v2
- model_version: 2.0.0
- threshold: 0.45
- artificial_delay_ms: 20

## 3. Описание Kubernetes-развёртывания

Созданы ресурсы:

- Namespace
- ConfigMap
- Secret
- Deployment v1
- Deployment v2
- Service общий
- Service v1
- Service v2
- HorizontalPodAutoscaler

Описание labels:

| Ресурс | Labels |
|---|---|
| Pod v1 | app=ml-service, version=v1 |
| Pod v2 | app=ml-service, version=v2 |
| Общий Service | selector app=ml-service |
| Service v1 | selector app=ml-service, version=v1 |
| Service v2 | selector app=ml-service, version=v2 |

Probes:

- readinessProbe: GET /health
- livenessProbe: GET /health

Resources:

| Параметр | Значение |
|---|---:|
| requests.cpu | 100m |
| requests.memory | 128Mi |
| limits.cpu | 500m |
| limits.memory | 512Mi |

## 4. Проверка отказоустойчивости

Эксперимент:

- число pod до удаления:
- удалённый pod:
- команда удаления:
- время восстановления:
- доступность сервиса во время восстановления:

Вывод:

## 5. HPA и нагрузочное тестирование

Параметры HPA:

| Параметр | Значение |
|---|---:|
| minReplicas | 2 |
| maxReplicas | 5 |
| target CPU utilization | 50% |

Инструмент нагрузки:

- hey / k6 / wrk / Locust:
- длительность:
- concurrency:
- endpoint:

Результаты:

| Этап | Replicas | CPU avg | RPS | Avg latency | p95 | p99 | Error rate |
|---|---:|---:|---:|---:|---:|---:|---:|
| До нагрузки | 2 | ... | ... | ... | ... | ... | ... |
| Во время нагрузки | ... | ... | ... | ... | ... | ... | ... |
| После нагрузки | ... | ... | ... | ... | ... | ... | ... |

Вывод по HPA:

## 6. Canary deployment

Конфигурации:

| Конфигурация | Реплики v1 | Реплики v2 | Ожидаемая доля v2 | Фактическая доля v2 | Error rate v2 |
|---|---:|---:|---:|---:|---:|
| Canary 10% | 9 | 1 | 10% | ... | ... |
| Canary 30% | 7 | 3 | 30% | ... | ... |

Как определялась версия:

- поле service_version в ответе /predict.

Решение:

- продвигать v2 / откатывать v2.

Обоснование:

## 7. A/B deployment

Схема:

- группа A направляется в service-v1;
- группа B направляется в service-v2.

Результаты:

| Группа | Версия | Requests | Avg latency | p95 | Error rate | Среднее предсказание |
|---|---|---:|---:|---:|---:|---:|
| A | v1 | ... | ... | ... | ... | ... |
| B | v2 | ... | ... | ... | ... | ... |

Отличие A/B от canary:

## 8. Rollback

Проблема в v2:

- высокий error rate / increased latency / BROKEN_MODE / другой сценарий.

Команды rollback:

    scripts/rollback.sh

Или вручную:

    kubectl scale deployment ml-service-v2 -n ml-lab --replicas=0
    kubectl scale deployment ml-service-v1 -n ml-lab --replicas=3

Результат проверки:

- /health:
- /metadata:
- /predict:

Время rollback:

## 9. Анализ и выводы

Ответить на вопросы:

1. Какие Kubernetes-ресурсы были использованы?
2. Как HPA реагировал на нагрузку?
3. Какие проблемы возникли при настройке HPA?
4. Чем canary отличается от A/B?
5. Какой способ rollback оказался наиболее удобным?
6. Что нужно добавить для production-уровня?

## 10. Production improvements

Возможные улучшения:

- использовать Ingress или Service Mesh для точного управления долей трафика;
- использовать Argo Rollouts или Flagger;
- добавить Prometheus и Grafana;
- добавить централизованные логи;
- добавить алерты по latency, error rate и saturation;
- хранить секреты в Vault или External Secrets Operator;
- использовать registry с подписанными образами;
- добавить canary analysis по метрикам;
- добавить мониторинг качества модели и data drift.
'''.strip() + "\n", encoding="utf-8")

print("REPORT.md создан.")

## 9. Финальная проверка созданных файлов

In [ ]:
# Выводим список созданных файлов.

for path in sorted(Path(".").rglob("*")):
    if path.is_file() and not path.name.endswith(".ipynb"):
        print(path)

## 10. Команды полного сценария

Ниже приведён рекомендуемый порядок выполнения лабораторной работы.

1. Собрать Docker-образы.
2. Загрузить образы в minikube/kind, если используется локальный кластер.
3. Развернуть v1.
4. Проверить `/health`, `/metadata`, `/predict`.
5. Проверить HPA под нагрузкой.
6. Развернуть v2 как canary.
7. Измерить фактическую долю трафика v2.
8. Провести A/B-тестирование.
9. Сломать v2 и выполнить rollback.
10. Заполнить REPORT.md.